# core

> What every tool result looks like, and the limits it is held to.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import json
from fastcore.basics import AttrDict, listify
from fastcore.xtras import str_diff

## Limits

Tool results consume model context. `MAX_TOOL_CHARS` limits one result. Factories accept the limit
as `mx` because the model sets the budget.

In [ ]:
#| export
MAX_TOOL_CHARS = 6000   # chars per tool result, budgeted for the smallest model
MAX_HITS = 20           # search hits one result carries
MAX_GREP_HITS = 60      # exact matches one `grep` returns. Part of the host contract
MAX_API = 200           # public names one `public_api` listing returns. Part of the same contract
MAX_FILE = 2_000_000    # bytes. A file larger than this is data, not source

In [ ]:
# a grep answers with more rows than a search, because an exact match is cheaper to read
assert MAX_GREP_HITS > MAX_HITS
assert MAX_TOOL_CHARS < 8_000 and MAX_FILE > MAX_TOOL_CHARS

Every search backend returns a `Hit` with the path, line, symbol and matching text.

In [ ]:
#| export
class Hit(AttrDict):
    "One search hit: path, line, symbol, text."
    def __init__(self, path, line=1, symbol='', text=''): super().__init__(path=path, line=line, symbol=symbol, text=text)
    def __repr__(self): return f'{self.path}:{self.line}  {self.symbol}  {self.text}'

In [ ]:
h = Hit('shalya/core.py', 12, 'clip', 'def clip(s, n=MAX_TOOL_CHARS, more=\'\'):')
h.path, h.line, h.symbol

In [ ]:
test_eq(repr(h), "shalya/core.py:12  clip  def clip(s, n=MAX_TOOL_CHARS, more=''):")

## Failure

Tool failures start with `ERROR: `. `failed` checks this prefix. Returning `err(...)` lets the model
inspect a failure without ending the turn.

In [ ]:
#| export
ERR = 'ERROR: '

class HostError(Exception): "A host refusal."

def host_err(e):
    "A caught exception, for a user-facing surface."
    return f'{type(e).__name__}: {e}'

def err(what, e=None):
    "One tool failure, spelled the way every other tool spells it."
    return f'{ERR}{what}' + (f': {host_err(e)}' if e is not None else '')

def failed(result):
    "Whether a tool result starts with `ERROR: `."
    return str(result or '').startswith(ERR)

In [ ]:
try: raise HostError('path is outside the open folders: /etc/passwd')
except HostError as e: shown = host_err(e)
shown

In [ ]:
test_eq(shown, 'HostError: path is outside the open folders: /etc/passwd')
test_eq(host_err(KeyError('nope')), "KeyError: 'nope'")
test_eq(err('could not read', HostError('no such file')), 'ERROR: could not read: HostError: no such file')
test_eq(failed(err('anything')), True)   # `err` and `failed` agree on the prefix, whatever it is

In [ ]:
try: json.loads('{not json}')
except Exception as e: msg = err('cannot read the edits', e)
msg

In [ ]:
test_eq(failed(msg), True)
test_eq(failed('ERROR was not the problem'), False)
test_eq(err('no such file'), 'ERROR: no such file')

## Clipping

`clip` truncates at a line boundary. `clip_lines` also reports the next line, which supports paging.

In [ ]:
#| export
def clip(s, n=MAX_TOOL_CHARS, more=''):
    "Truncate a tool result to `n` chars. A caller with a way to resume passes it as `more`."
    s, n = str(s), max(1, int(n))   # a caller subtracting a margin can arrive at zero or less
    if len(s) <= n: return s
    cut = s[:n]
    nl = cut.rfind('\n')
    if nl > n * 0.6: cut = cut[:nl]
    note = f'[truncated: {len(cut)} of {len(s)} chars shown'
    return cut + f'…{note}. {more}]' if more else cut + f'…{note}]'

In [ ]:
print(clip('one\ntwo\nthree\nfour\nfive\n', 12))

In [ ]:
test_eq(clip('short', 12), 'short')
test_eq(clip('one\ntwo\nthree\nfour', 12).splitlines()[0], 'one')
test_eq(clip('hello', 5), 'hello')
# a budget of zero or less kept the last of the string and reported "0 of 5 chars shown"
assert clip('hello', 0).startswith('h') and '1 of 5 chars shown' in clip('hello', 0), clip('hello', 0)
assert clip('hello', -200).startswith('h'), clip('hello', -200)

A line longer than the budget is truncated by character count. The notice reports the original and
shown lengths without offering the same line as a resume point.

In [ ]:
#| export
def clip_lines(lines, start=1, n=MAX_TOOL_CHARS, more='', empty='(nothing)'):
    "Render `lines` within the budget, and say which line to resume from."
    lines = listify(lines)
    if not lines: return empty
    out, used = [], 0
    for i, line in enumerate(lines):
        line = str(line)
        if used + len(line) + 1 > n:
            if out:
                rest = len(lines) - i
                tail = f'\n…[{rest} more line(s) not shown'
                hint = more.format(next=start + i) if '{next}' in more else more
                return '\n'.join(out) + (f'{tail}. {hint}]' if hint else f'{tail}]')
            keep, rest = max(1, n - 1), len(lines) - 1
            more_lines = f', and {rest} more line(s) not shown' if rest else ''
            return line[:keep] + f'\n…[line {start} is {len(line)} chars; {keep} shown{more_lines}]'
        out.append(line); used += len(line) + 1
    return '\n'.join(out)

In [ ]:
print(clip_lines(['def a(): pass', 'def b(): pass', 'def c(): pass'], n=30,
                more='read from line {next}'))

In [ ]:
test_eq(clip_lines([], empty='(no matches)'), '(no matches)')
test_eq(clip_lines(['a', 'b'], n=100), 'a\nb')
long = clip_lines(['x' * 50], n=20)
assert long.startswith('x' * 19) and '50 chars' in long, long
# the resume point is the line the reader has not seen yet, counted from `start`
test_eq(clip_lines(['def a(): pass'] * 3, n=30, more='read from line {next}'),
        'def a(): pass\ndef a(): pass\n…[1 more line(s) not shown. read from line 3]')
test_eq(clip_lines(['def a(): pass'] * 3, start=10, n=30, more='read from line {next}'),
        'def a(): pass\ndef a(): pass\n…[1 more line(s) not shown. read from line 12]')

## Edits

Tool calls carry edits as JSON. `cmds` parses exhash commands. `edits` parses exact-text
replacements. Both reject ambiguous input with an actionable error.

In [ ]:
#| export
def cmds(commands):
    "Models emit JSON and exhash wants tuples, nested ones included: `[[...]]` becomes `[(...)]`."
    if isinstance(commands, str): commands = json.loads(commands)
    if not isinstance(commands, list): raise ValueError('commands must be a JSON list of command arrays')
    def _t(c):
        if not isinstance(c, (list, tuple)): raise ValueError(f'each command must be an array, got {type(c).__name__}')
        return tuple(_t(x) if isinstance(x, (list, tuple)) else x for x in c)
    return [_t(c) for c in commands]

In [ ]:
cmds('[["12|a1b2|", "s", "old", "new"], ["30|9f3c|", "a", "appended"]]')

In [ ]:
test_eq(cmds([['1|ab|', 'd']]), [('1|ab|', 'd')])
test_fail(lambda: cmds('{"not": "a list"}'), contains='must be a JSON list')
test_fail(lambda: cmds('["bare string"]'), contains='must be an array')

`edits` takes the exact-text form in any of its three spellings. `apply_edits` holds the refusals.
Each one names the edit that is wrong and what to do about it. A match that is not unique needs
more context. Two edits over the same span need merging.

In [ ]:
#| export
def edits(es):
    "A JSON string, `{'oldText','newText'}` dicts, or `[old, new]` pairs: all three are unambiguous."
    if isinstance(es, str): es = json.loads(es)
    if isinstance(es, dict): es = [es]
    if not isinstance(es, (list, tuple)): raise ValueError('edits must be a JSON array')
    out = []
    for e in es:
        if isinstance(e, dict):
            if 'oldText' not in e or 'newText' not in e: raise ValueError("each edit needs 'oldText' and 'newText'")
            out.append((str(e['oldText']), str(e['newText'])))
        elif isinstance(e, (list, tuple)) and len(e) == 2: out.append((str(e[0]), str(e[1])))
        else: raise ValueError('each edit must be {"oldText":…,"newText":…} or [old, new]')
    return out

def apply_edits(text, es):
    "Apply exact-text edits to `text`, or raise saying which one is wrong and why."
    spans = []
    for i, (old, new) in enumerate(es, 1):
        if not old: raise ValueError(f'edit {i}: oldText is empty; use create_file to write a whole file')
        n = text.count(old)
        if n == 0: raise ValueError(f'edit {i}: oldText not found. It must match the file exactly, including indentation. Re-read the file and try again')
        if n > 1: raise ValueError(f'edit {i}: oldText matches {n} places. Include more surrounding lines so it matches exactly one')
        at = text.index(old)
        spans.append((at, at + len(old), new, i))
    spans.sort()
    for (s1, e1, _, i1), (s2, _, _, i2) in zip(spans, spans[1:]):
        if s2 < e1: raise ValueError(f'edits {i1} and {i2} overlap; merge them into one edit')
    out, at = [], 0
    for s, e, new, _ in spans:
        out.append(text[at:s])
        out.append(new)
        at = e
    out.append(text[at:])
    return ''.join(out)

In [ ]:
before = 'def greet(name):\n    return "hello " + name\n'
after = apply_edits(before, edits('[{"oldText": "hello", "newText": "howdy"}]'))
after

In [ ]:
test_eq(edits([['a', 'b']]), [('a', 'b')])
test_eq(edits({'oldText': 'a', 'newText': 'b'}), [('a', 'b')])
test_eq(after, 'def greet(name):\n    return "howdy " + name\n')

The three refusals, on real text. Each message tells the model what to change rather than that it
failed.

In [ ]:
test_fail(lambda: apply_edits(before, [('name', 'who')]), contains='matches 2 places')
test_fail(lambda: apply_edits(before, [('nowhere', 'x')]), contains='not found')
test_fail(lambda: apply_edits(before, [('def greet', 'def hi'), ('greet(name)', 'hi(n)')]),
          contains='overlap')

Edit approvals carry a unified diff.

In [ ]:
#| export
def diff_text(before, after, path='file'):
    "Return a unified diff."
    return str_diff(before, after, n=2, names=(f'a/{path}', f'b/{path}'))

In [ ]:
print(diff_text(before, after, 'greet.py'))

In [ ]:
assert '-    return "hello " + name' in diff_text(before, after)
assert diff_text(before, before) == ''

## Which tools write

Writes require approval. `@writes` marks a callable; `WRITE_TOOLS` provides the same fact when a
caller has only its name. The tests require both representations to agree.

In [ ]:
#| export
def writes(f):
    "Mark a tool as one that changes something. `Approvals` puts these in front of a person."
    f.writes = True
    return f

def is_write(t):
    "Whether `t` is a tool that changes something."
    return bool(getattr(t, 'writes', False))

def acts(f):
    "Mark a tool that acts without writing a file the user owns."
    f.acts = True
    return f

def has_effect(t):
    "Whether `t` acts. Orthogonal to `is_write`: these are the effects approval does not gate."
    return bool(getattr(t, 'acts', False))

#: Rehearsing a merge is not approving one, so the git tools split before the write set uses them.
GIT_READ_TOOLS = ('git_status', 'git_divergence', 'git_rebase_preview')
GIT_WRITE_TOOLS = frozenset({'git_remote', 'git_checkout'})
GIT_TOOLS = (*GIT_READ_TOOLS, *sorted(GIT_WRITE_TOOLS))

#: The same fact as `is_write`, by name, for callers that only have a name.
WRITE_TOOLS = frozenset({'edit_file', 'replace_text', 'create_file', 'edit_cell', 'add_cell',
                         'run_python', 'run_shell', 'memory_forget', 'create_skill',
                         'cancel_watch', 'add_root'}) | GIT_WRITE_TOOLS

#: The same fact as `has_effect`, by name. `read_only(effects=False)` withholds these.
ACTING_TOOLS = frozenset({'inspect_python', 'api_call', 'generate_image', 'research', 'watch_url',
                          'set_reminder'})

In [ ]:
@writes
def create_file(path, text): return f'wrote {path}'

def view_file(path): return 'contents'

is_write(create_file), is_write(view_file)

In [ ]:
test_eq(is_write(create_file), True)
test_eq(is_write(view_file), False)
test_eq(has_effect(create_file), False)                # writing and acting are separate marks
test_eq(len(GIT_TOOLS), len(set(GIT_TOOLS)))           # a name in both halves would be offered twice
assert not (WRITE_TOOLS & ACTING_TOOLS), 'a tool either writes or acts, never both'
assert set(GIT_WRITE_TOOLS) <= WRITE_TOOLS

## What a tool just did

A docstring is read by the model before it chooses a tool. A summary is read by a person afterwards
and is made from that call's arguments, so `View a.py:4-9` cannot be derived from one. `@summary`
marks the line beside the tool; `SUMMARIES` answers the same question from a name alone.

In [ ]:
#| export
def one_line(v, n=90):
    "One line of a value, short enough to sit in a list."
    t = ' '.join(str(v or '').split())
    return t if len(t) <= n else t[:n - 1] + '…'

SUMMARIES = {}

def summary(fn):
    "Mark the one line a person reads after this tool runs. `fn` is given the call's arguments."
    def _mark(t):
        t.summary = fn
        SUMMARIES[t.__name__] = fn
        return t
    return _mark

def summarise(tool, args=None):
    "The imperative one-liner for a call: what a person would say they just did."
    a = args if isinstance(args, dict) else {}
    nm = tool if isinstance(tool, str) else getattr(tool, '__name__', '')
    fn = getattr(tool, 'summary', None) or SUMMARIES.get(nm)
    if fn is not None:
        try: return fn(a)
        except Exception: pass
    return f'{nm}({", ".join(f"{k}={one_line(v, 30)!r}" for k, v in a.items())})'

In [ ]:
@summary(lambda a: f'View {a.get("path","")}' + (f':{a["start"]}-{a["end"]}' if a.get('start') else ''))
def view_file(path, start=None, end=None): return 'contents'

summarise(view_file, {'path': 'a.py', 'start': 4, 'end': 9}), summarise(view_file, {'path': 'a.py'})

In [ ]:
test_eq(summarise(view_file, {'path': 'a.py', 'start': 4, 'end': 9}), 'View a.py:4-9')
test_eq(summarise(view_file, {'path': 'a.py'}), 'View a.py')
test_eq(summarise('view_file', {'path': 'a.py'}), 'View a.py')   # by name, for a caller holding one
test_eq(one_line('  two   words  '), 'two words')
test_eq(one_line('x' * 12, 8), 'xxxxxxx…')
# a tool nobody marked still says what was called, rather than nothing
test_eq(summarise(lambda: None, {}), '<lambda>()')
def unmarked(path): return ''
test_eq(summarise(unmarked, {'path': 'a.py'}), "unmarked(path='a.py')")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()